# Préparation des données BAAC 2020–2024

Objectif : regrouper les fichiers annuels en 4 jeux de données consolidés (`Caract`, `Lieux`, `Vehicules`, `Usagers`) après vérification de leur compatibilité.

## 1. Préparation des fichiers `Caract`

Les fichiers `Caract` 2020 à 2024 sont comparés avant leur regroupement afin de vérifier qu'ils peuvent être concaténés.

### 1.1. Chargement des fichiers `Caract`

Chargement des cinq fichiers annuels afin de comparer leur structure avant concaténation.

In [160]:
# 1.1. Chargement des fichiers Caract

import pandas as pd
import os

df_caract_2020 = pd.read_csv("data/raw/2020/caract-2020.csv", sep=";")
df_caract_2021 = pd.read_csv("data/raw/2021/caract-2021.csv", sep=";")
df_caract_2022 = pd.read_csv("data/raw/2022/caract-2022.csv", sep=";")
df_caract_2023 = pd.read_csv("data/raw/2023/caract-2023.csv", sep=";")
df_caract_2024 = pd.read_csv("data/raw/2024/caract-2024.csv", sep=";")

In [161]:
# 1.2. Vérification de la structure des fichiers Caract

for annee, df in zip(
    [2020, 2021, 2022, 2023, 2024],
    [df_caract_2020, df_caract_2021, df_caract_2022, df_caract_2023, df_caract_2024]
):
    print(f"{annee} : {df.shape}")
    print(df.columns.tolist())

2020 : (47744, 15)
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
2021 : (56518, 15)
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
2022 : (55302, 15)
['Accident_Id', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
2023 : (54822, 15)
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']
2024 : (54402, 15)
['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']


### 1.3. Contrôle des identifiants

L'unicité et les valeurs manquantes des identifiants sont vérifiées pour chaque année avant leur harmonisation.

In [162]:
# 1.3. Contrôle des identifiants

fichiers_caract = {
    2020: (df_caract_2020, "Num_Acc"),
    2021: (df_caract_2021, "Num_Acc"),
    2022: (df_caract_2022, "Accident_Id"),
    2023: (df_caract_2023, "Num_Acc"),
    2024: (df_caract_2024, "Num_Acc")
}

for annee, (df, identifiant) in fichiers_caract.items():
    print(f"\n--- {annee} ---")
    print("Nombre de lignes :", len(df))
    print("Identifiants uniques :", df[identifiant].nunique())
    print("Valeurs manquantes :", df[identifiant].isna().sum())


--- 2020 ---
Nombre de lignes : 47744
Identifiants uniques : 47744
Valeurs manquantes : 0

--- 2021 ---
Nombre de lignes : 56518
Identifiants uniques : 56518
Valeurs manquantes : 0

--- 2022 ---
Nombre de lignes : 55302
Identifiants uniques : 55302
Valeurs manquantes : 0

--- 2023 ---
Nombre de lignes : 54822
Identifiants uniques : 54822
Valeurs manquantes : 0

--- 2024 ---
Nombre de lignes : 54402
Identifiants uniques : 54402
Valeurs manquantes : 0


#### Résultat

Pour chaque année, le nombre d'identifiants uniques est égal au nombre de lignes et aucune valeur manquante n'est présente.

Les identifiants sont donc complets et uniques dans chaque fichier annuel.

**Décision :** harmoniser le nom `Accident_Id` de 2022 en `Num_Acc` avant la concaténation.

### 1.4. Vérification des types de données

Les types des colonnes sont comparés entre les cinq années afin d'identifier d'éventuelles différences à harmoniser avant la concaténation.

In [163]:
# 1.4. Vérification des types de données

types_caract = pd.DataFrame({
    "2020": df_caract_2020.dtypes,
    "2021": df_caract_2021.dtypes,
    "2022": df_caract_2022.rename(
        columns={"Accident_Id": "Num_Acc"}
    ).dtypes,
    "2023": df_caract_2023.dtypes,
    "2024": df_caract_2024.dtypes
})

types_caract

,2020,2021,2022,2023,2024
Num_Acc,int64,int64,int64,int64,int64
jour,int64,int64,int64,int64,int64
mois,int64,int64,int64,int64,int64
an,int64,int64,int64,int64,int64
hrmn,str,str,str,str,str
lum,int64,int64,int64,int64,int64
dep,str,str,str,str,str
com,str,str,str,str,str
agg,int64,int64,int64,int64,int64
int,int64,int64,int64,int64,int64


#### Résultat

Les types de données sont identiques pour les cinq années après alignement du nom de l'identifiant 2022.

**Décision :** renommer `Accident_Id` en `Num_Acc` dans le fichier 2022 avant concaténation.

### 1.5. Harmonisation de l'identifiant

La colonne `Accident_Id` de 2022 est renommée `Num_Acc` afin d'obtenir la même structure pour les cinq années.

In [164]:
# 1.5. Harmonisation de l'identifiant

df_caract_2022 = df_caract_2022.rename(
    columns={"Accident_Id": "Num_Acc"}
)

In [165]:
# Vérification après harmonisation

colonnes_identiques = all(
    df.columns.equals(df_caract_2020.columns)
    for df in [
        df_caract_2021,
        df_caract_2022,
        df_caract_2023,
        df_caract_2024
    ]
)

colonnes_identiques

True

#### Résultat

Les colonnes sont désormais identiques pour les cinq années.

**Décision :** les fichiers `Caract` peuvent être concaténés.

### 1.6. Concaténation des fichiers `Caract`

Les cinq fichiers compatibles sont regroupés dans un seul DataFrame couvrant la période 2020–2024.

In [166]:
# 1.6. Concaténation des fichiers Caract

df_caract = pd.concat(
    [
        df_caract_2020,
        df_caract_2021,
        df_caract_2022,
        df_caract_2023,
        df_caract_2024
    ],
    ignore_index=True
)

df_caract.shape

(268788, 15)

### 1.7. Vérification de la concaténation

Le nombre d'observations par année est contrôlé afin de vérifier que toutes les données ont été conservées lors de la concaténation.

In [167]:
# 1.7. Vérification de la concaténation

df_caract["an"].value_counts().sort_index()

an
2020    47744
2021    56518
2022    55302
2023    54822
2024    54402
Name: count, dtype: int64

#### Résultat

Les **268 788 observations** sont bien réparties entre les cinq années avec les mêmes effectifs que dans les fichiers d'origine.

**Décision :** la concaténation est validée et `df_caract` devient le DataFrame de travail pour les données `Caract` 2020–2024.

### 1.8. Sauvegarde de la version consolidée

Le DataFrame `Caract` 2020–2024 est sauvegardé avant nettoyage afin de conserver une version consolidée intermédiaire.

In [168]:
# 1.8. Sauvegarde de la version consolidée

from pathlib import Path

Path("data/interim").mkdir(parents=True, exist_ok=True)

df_caract.to_csv(
    "data/interim/caract_2020_2024.csv",
    sep=";",
    index=False
)
df_caract.head()

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202000000001,7,3,2020,16:55,1,91,91657,2,3,1,3,HENRI BARBUSSE (AVENUE),"48,7053500","2,4384100"
1,202000000002,7,3,2020,08:35,2,91,91657,2,9,7,6,MOUSSEAUX(CHEMIN),"48,6900000","2,4100000"
2,202000000003,7,3,2020,13:30,1,91,91174,2,2,1,3,CARNOT(AVENUE),"48,6106700","2,4758200"
3,202000000004,7,3,2020,18:50,5,91,91215,2,1,1,6,VICTOR HUGO (AVENUE),"48,6978200","2,5244600"
4,202000000005,7,3,2020,11:00,1,77,77181,1,6,1,2,LAGNY (RUE DE ) - D35,"48,8286457","2,7059707"


## 2. Contrôle qualité de `Caract`

Les principaux contrôles qualité permettent d'identifier les traitements nécessaires avant nettoyage.

### 2.1. Contrôle des doublons

Vérification de la présence éventuelle de lignes dupliquées.

In [169]:
# 2.1. Contrôle des doublons

df_caract.duplicated().sum()

np.int64(0)

#### Résultat

Aucun doublon complet n'a été détecté.

**Décision :** aucune ligne ne sera supprimée pour cause de doublon.

### 2.2. Contrôle des valeurs manquantes

Identification des variables contenant des valeurs manquantes avant d'étudier leur signification et leur traitement.

In [170]:
# 2.2. Contrôle des valeurs manquantes

df_caract.isna().sum()

Num_Acc       0
jour          0
mois          0
an            0
hrmn          0
lum           0
dep           0
com           0
agg           0
int           0
atm           0
col           0
adr        5862
lat           0
long          0
dtype: int64

#### Résultat

Seule la variable `adr` contient des valeurs manquantes : **5 862**, soit environ **2,18 %** des observations.

Les autres informations géographiques (`dep`, `com`, `lat` et `long`) sont renseignées pour toutes les observations.

**Décision :**  **`adr`** : 5 862 valeurs `NaN` et deux valeurs `.` ont été identifiées. Les observations seront conservées car les autres informations géographiques (`dep`, `com`, `lat` et `long`) permettent de maintenir une information de localisation exploitable. Les valeurs `.` seront remplacées par `NaN` lors du nettoyage ; aucune imputation d'adresse ne sera réalisée.

### 2.3. Contrôle des variables catégorielles codées

Les modalités des principales variables catégorielles sont contrôlées afin de vérifier leur cohérence avec le dictionnaire BAAC et d'identifier les éventuelles valeurs non renseignées.

In [171]:
# 2.3. Contrôle des variables catégorielles codées

colonnes_codees = ["lum", "agg", "int", "atm", "col"]

for colonne in colonnes_codees:
    print(df_caract[colonne].value_counts().sort_index())
    print()

lum
-1         9
 1    179407
 2     17748
 3     27376
 4      2693
 5     41555
Name: count, dtype: int64

agg
1     98314
2    170474
Name: count, dtype: int64

int
-1        14
 1    171766
 2     32663
 3     30062
 4      5869
 5      1563
 6     11773
 7      2675
 8       614
 9     11789
Name: count, dtype: int64

atm
-1        25
 1    212417
 2     30062
 3      6142
 4       806
 5      1948
 6       791
 7      4746
 8     10642
 9      1209
Name: count, dtype: int64

col
-1     1617
 1    28088
 2    35428
 3    81516
 4     9373
 5     7679
 6    78221
 7    26866
Name: count, dtype: int64



#### Résultat

Les modalités observées sont cohérentes avec les codes attendus.

Des valeurs `-1`, correspondant à des informations non renseignées, sont présentes dans `lum` (9), `int` (14), `atm` (25) et `col` (1 617). La variable `agg` ne présente pas de valeur non renseignée.

**Décision :** lors du nettoyage, les valeurs `-1` de `lum`, `int`, `atm` et `col` seront remplacées par `NaN` afin de distinguer les informations non renseignées des modalités réellement observées. Les lignes concernées seront conservées.

### 2.4. Contrôle des variables temporelles

Vérification de la validité des dates et des heures afin d'identifier d'éventuelles valeurs impossibles ou mal renseignées.

In [172]:
# 2.4. Contrôle des variables temporelles

dates_test = pd.to_datetime(
    df_caract[["an", "mois", "jour"]].rename(
        columns={"an": "year", "mois": "month", "jour": "day"}
    ),
    errors="coerce"
)

dates_test.isna().sum()

np.int64(0)

#### Résultat

Aucune date invalide n'a été détectée parmi les **268 788 observations**. Les combinaisons `an`, `mois` et `jour` correspondent toutes à des dates valides.

**Décision :** aucune correction des valeurs n'est nécessaire. Lors du nettoyage, une variable `date_accident` au format date sera créée à partir de `an`, `mois` et `jour` afin de faciliter les analyses temporelles.

#### Contrôle des heures

Vérification du format et de la validité des heures renseignées dans la variable `hrmn`.

In [173]:
# Contrôle de la variable hrmn

heures_test = pd.to_datetime(
    df_caract["hrmn"],
    format="%H:%M",
    errors="coerce"
)

heures_test.isna().sum()

np.int64(0)

#### Résultat

Aucune heure invalide n'a été détectée dans la variable `hrmn`.

**Décision :** aucune correction n'est nécessaire. La variable `hrmn` sera conservée au format `HH:MM` afin de préserver l'heure et les minutes de l'accident.

### 2.5. Contrôle des variables géographiques

Vérification de la cohérence des codes géographiques et des coordonnées afin d'identifier les éventuelles valeurs nécessitant un traitement.

#### Contrôle des codes département et commune

Vérification du format des variables `dep` et `com`, qui correspondent à des codes géographiques et non à des variables quantitatives.

In [174]:
# 2.5. Contrôle des codes département et commune

print("dep :", df_caract["dep"].str.len().value_counts().sort_index())
print()
print("com :", df_caract["com"].str.len().value_counts().sort_index())

dep : dep
2    253740
3     15048
Name: count, dtype: int64

com : com
3         1
5    268787
Name: count, dtype: int64


#### Résultat

Les codes `dep` présentent des longueurs de 2 ou 3 caractères. La variable `com` contient presque exclusivement des codes de 5 caractères, à l'exception d'une observation de longueur 3.

Cette observation doit être examinée avant de déterminer s'il s'agit d'une anomalie ou d'un format particulier.

In [175]:
# Identification du code commune de longueur inhabituelle

df_caract.loc[
    df_caract["com"].str.len() != 5,
    ["Num_Acc", "an", "dep", "com"]
]

,Num_Acc,an,dep,com
104780,202200000519,2022,14,N/C


#### Contrôle des valeurs textuelles non renseignées

Recherche de valeurs textuelles particulières pouvant représenter une information non renseignée mais non détectée comme `NaN`.

In [176]:
# Contrôle des valeurs textuelles non renseignées

valeurs_non_renseignees = ["N/C", "NC", "N/A", ".", ""]

for colonne in df_caract.select_dtypes(include="object").columns:
    masque = df_caract[colonne].isin(valeurs_non_renseignees)
    
    if masque.any():
        print(f"{colonne} :")
        print(df_caract.loc[masque, colonne].value_counts())
        print()

com :
com
N/C    1
Name: count, dtype: int64

adr :
adr
.    2
Name: count, dtype: int64



C:\Users\nabil\AppData\Local\Temp\ipykernel_28180\86848016.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for colonne in df_caract.select_dtypes(include="object").columns:


#### Résultat

Deux formes supplémentaires de valeurs non renseignées ont été identifiées : une valeur `N/C` dans `com` et deux valeurs `.` dans `adr`.

Ces valeurs ne sont pas détectées par `isna()` car elles sont enregistrées comme du texte.

**Décision :** lors du nettoyage, `N/C` dans `com` et `.` dans `adr` seront remplacés par `NaN`. Les observations concernées seront conservées.

#### Contrôle des coordonnées géographiques

Vérification de la possibilité de convertir `lat` et `long` en valeurs numériques avant leur utilisation comme coordonnées géographiques.

In [177]:
# Contrôle de la conversion des coordonnées

lat_test = pd.to_numeric(
    df_caract["lat"].str.replace(",", ".", regex=False),
    errors="coerce"
)

long_test = pd.to_numeric(
    df_caract["long"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Latitude non convertible :", lat_test.isna().sum())
print("Longitude non convertible :", long_test.isna().sum())

Latitude non convertible : 0
Longitude non convertible : 0


#### Contrôle des plages de coordonnées

Vérification que les coordonnées respectent les plages géographiques possibles :
- latitude comprise entre **-90° et 90°** ;
- longitude comprise entre **-180° et 180°**.

Toute valeur située en dehors de ces intervalles serait considérée comme géographiquement impossible et nécessiterait une investigation.

In [178]:
# Contrôle des plages de coordonnées

print("Latitude min :", lat_test.min())
print("Latitude max :", lat_test.max())

print("Longitude min :", long_test.min())
print("Longitude max :", long_test.max())

Latitude min : -23.868998
Latitude max : 51.07874
Longitude min : -178.15809
Longitude max : 168.106549


#### Résultat

Toutes les valeurs de `lat` et `long` sont convertibles en valeurs numériques.

Les coordonnées observées respectent également les plages géographiques possibles :
- latitude : de **-23,868998° à 51,078740°**, comprise dans l'intervalle valide **[-90° ; 90°]** ;
- longitude : de **-178,158090° à 168,106549°**, comprise dans l'intervalle valide **[-180° ; 180°]**.

Aucune coordonnée géographiquement impossible n'a été détectée.

**Décision :** aucune observation ne sera supprimée. Lors du nettoyage, `lat` et `long` seront converties en variables numériques en remplaçant la virgule décimale par un point.

### 2.6. Synthèse des décisions de nettoyage

À l'issue des contrôles qualité, les décisions suivantes sont retenues :

- **Doublons** : aucun doublon complet détecté ; aucune suppression nécessaire.
- **`adr`** : conserver les valeurs manquantes et remplacer les deux valeurs `.` par `NaN`.
- **`lum`, `int`, `atm`, `col`** : remplacer les valeurs `-1` correspondant à des informations non renseignées par `NaN`.
- **`com`** : remplacer la valeur `N/C` par `NaN` ; conserver l'observation concernée.
- **Dates** : aucune date invalide ; créer une variable `date_accident` à partir de `an`, `mois` et `jour`.
- **`hrmn`** : aucune heure invalide détectée ; conserver la variable au format `HH:MM` afin de préserver l'heure et les minutes de l'accident. Aucun traitement supplémentaire n'est nécessaire.
- **`dep` et `com`** : conserver leur nature de codes géographiques et ne pas les traiter comme des variables quantitatives.
- **`lat` et `long`** : toutes les valeurs sont convertibles et respectent les plages géographiques possibles ; les convertir en variables numériques.
- **Observations** : aucune ligne ne sera supprimée à ce stade, les informations manquantes identifiées ne justifiant pas la suppression complète d'un accident.

## 3. Nettoyage de `Caract`

Application des décisions prises lors du contrôle qualité afin d'obtenir des données cohérentes et directement exploitables pour les prochaines étapes du projet.

### 3.1. Traitement des valeurs non renseignées

Les valeurs `-1` identifiées dans `lum`, `int`, `atm` et `col` sont remplacées par des valeurs manquantes afin de ne pas les considérer comme de véritables modalités lors des analyses.

In [179]:
# 3.1. Traitement des valeurs non renseignées

colonnes_non_renseignees = ["lum", "int", "atm", "col"]

df_caract[colonnes_non_renseignees] = (
    df_caract[colonnes_non_renseignees]
    .replace(-1, pd.NA)
    .astype("Int64")
)

In [180]:
# Vérification du traitement des valeurs non renseignées

df_caract[colonnes_non_renseignees].isna().sum()

lum       9
int      14
atm      25
col    1617
dtype: int64

#### Résultat

Les valeurs `-1` ont été correctement remplacées par des valeurs manquantes dans `lum` (9), `int` (14), `atm` (25) et `col` (1 617).

Le traitement est conforme aux anomalies identifiées lors du contrôle qualité.

### 3.2. Traitement des valeurs textuelles non renseignées

Les valeurs `N/C` dans `com` et `.` dans `adr`, identifiées comme des informations non renseignées lors du contrôle qualité, sont remplacées par des valeurs manquantes.

In [181]:
# 3.2. Traitement des valeurs textuelles non renseignées

df_caract["com"] = df_caract["com"].replace("N/C", pd.NA)

df_caract["adr"] = df_caract["adr"].replace(".", pd.NA)

In [182]:
# Vérification du traitement des valeurs textuelles non renseignées

df_caract[["com", "adr"]].isna().sum()

com       1
adr    5864
dtype: int64

#### Résultat

Les valeurs textuelles non renseignées ont été correctement traitées : `com` contient désormais 1 valeur manquante et `adr` 5 864.

Les observations concernées sont conservées, les autres informations disponibles permettant de maintenir leur exploitation.

### 3.3. Conversion des coordonnées géographiques

Les variables `lat` et `long` sont converties en valeurs numériques afin de permettre leur utilisation dans les analyses et visualisations géographiques.

In [183]:
# 3.3. Conversion des coordonnées géographiques

df_caract["lat"] = pd.to_numeric(
    df_caract["lat"].str.replace(",", ".", regex=False)
)

df_caract["long"] = pd.to_numeric(
    df_caract["long"].str.replace(",", ".", regex=False)
)

In [184]:
# Vérification des types après conversion

df_caract[["lat", "long"]].dtypes

lat     float64
long    float64
dtype: object

#### Résultat

Les variables `lat` et `long` ont été correctement converties en type numérique (`float64`).

Elles sont désormais directement exploitables pour les analyses et visualisations géographiques.

### 3.4. Création de la date de l'accident

Une variable `date_accident` est créée à partir de `an`, `mois` et `jour` afin de faciliter les analyses temporelles.

In [185]:
# 3.4. Création de la date de l'accident

df_caract["date_accident"] = pd.to_datetime(
    df_caract[["an", "mois", "jour"]].rename(
        columns={"an": "year", "mois": "month", "jour": "day"}
    )
)

In [186]:
# Vérification de la variable date_accident

print("Type :", df_caract["date_accident"].dtype)
print()
print(df_caract[["an", "mois", "jour", "date_accident"]].head())

Type : datetime64[us]

     an  mois  jour date_accident
0  2020     3     7    2020-03-07
1  2020     3     7    2020-03-07
2  2020     3     7    2020-03-07
3  2020     3     7    2020-03-07
4  2020     3     7    2020-03-07


#### Résultat

La variable `date_accident` a été correctement créée à partir de `an`, `mois` et `jour`.

Elle est conservée au format date afin de faciliter les analyses temporelles et son utilisation ultérieure dans Power BI.

### 3.5. Renommage des variables

Les variables descriptives sont renommées avec des intitulés explicites afin de faciliter leur compréhension et leur utilisation dans les prochaines étapes du projet.

L'identifiant `Num_Acc` conserve son nom d'origine afin de maintenir une clé commune entre les différentes tables BAAC.

In [187]:
# 3.5. Renommage des variables

df_caract = df_caract.rename(
    columns={
        "an": "annee",
        "hrmn": "heure_minute",
        "lum": "luminosite",
        "dep": "departement",
        "com": "commune",
        "agg": "agglomeration",
        "int": "intersection",
        "atm": "conditions_atmospheriques",
        "col": "type_collision",
        "adr": "adresse",
        "lat": "latitude",
        "long": "longitude"
    }
)

df_caract.columns.tolist()

['Num_Acc',
 'jour',
 'mois',
 'annee',
 'heure_minute',
 'luminosite',
 'departement',
 'commune',
 'agglomeration',
 'intersection',
 'conditions_atmospheriques',
 'type_collision',
 'adresse',
 'latitude',
 'longitude',
 'date_accident']

## 4. Validation finale et export de `Caract`

Une validation finale est réalisée afin de vérifier que le nettoyage n'a pas modifié le nombre d'observations et que le jeu de données est prêt à être enregistré dans le dossier `processed`.

In [188]:
# 4.1. Vérification de la structure après nettoyage

df_caract.shape

(268788, 16)

#### Résultat

Le jeu de données nettoyé contient **268 788 observations et 16 variables**.

Le nombre d'observations est identique à celui obtenu après la consolidation des données 2020–2024. Aucune ligne n'a donc été supprimée lors du nettoyage.

Les 16 variables correspondent aux **15 variables d'origine du fichier Caract**, auxquelles s'ajoute la variable `date_accident` créée pour faciliter les analyses temporelles.

In [189]:
# 4.2. Vérification finale de l'identifiant

print("Nombre de lignes :", len(df_caract))
print("Nombre de Num_Acc uniques :", df_caract["Num_Acc"].nunique())
print("Num_Acc manquants :", df_caract["Num_Acc"].isna().sum())

Nombre de lignes : 268788
Nombre de Num_Acc uniques : 268788
Num_Acc manquants : 0


#### Résultat

Les **268 788 observations** possèdent chacune un identifiant `Num_Acc` unique et aucune valeur manquante n'est présente dans cette variable.

`Num_Acc` peut donc être utilisé comme identifiant unique des accidents dans la table `Caract` et permettra ensuite de relier cette table aux autres tables BAAC.

In [190]:
# 4.3. Vérification finale des types

df_caract.dtypes

Num_Acc                               int64
jour                                  int64
mois                                  int64
annee                                 int64
heure_minute                            str
luminosite                            Int64
departement                             str
commune                                 str
agglomeration                         int64
intersection                          Int64
conditions_atmospheriques             Int64
type_collision                        Int64
adresse                                 str
latitude                            float64
longitude                           float64
date_accident                datetime64[us]
dtype: object

#### Résultat

Les types des variables après nettoyage sont cohérents avec leur utilisation :

- les variables catégorielles codées pouvant contenir des valeurs manquantes (`luminosite`, `intersection`, `conditions_atmospheriques`, `type_collision`) sont au format entier nullable `Int64` ;
- les codes géographiques `departement` et `commune` sont conservés comme variables textuelles ;
- les coordonnées `latitude` et `longitude` sont au format numérique ;
- `heure_minute` conserve l'heure complète au format `HH:MM` ;
- `date_accident` est correctement reconnue comme variable temporelle.

Le jeu de données présente donc des types adaptés aux prochaines étapes d'analyse.

In [191]:
# 4.4. Export du jeu de données nettoyé

Path("data/processed").mkdir(parents=True, exist_ok=True)

df_caract.to_csv(
    "data/processed/caract_2020_2024_clean.csv",
    sep=";",
    index=False,
    date_format="%Y-%m-%d"
)
df_caract.head()

,Num_Acc,jour,mois,annee,heure_minute,luminosite,departement,commune,agglomeration,intersection,conditions_atmospheriques,type_collision,adresse,latitude,longitude,date_accident
0,202000000001,7,3,2020,16:55,1,91,91657,2,3,1,3,HENRI BARBUSSE (AVENUE),48.705350,2.438410,2020-03-07
1,202000000002,7,3,2020,08:35,2,91,91657,2,9,7,6,MOUSSEAUX(CHEMIN),48.690000,2.410000,2020-03-07
2,202000000003,7,3,2020,13:30,1,91,91174,2,2,1,3,CARNOT(AVENUE),48.610670,2.475820,2020-03-07
3,202000000004,7,3,2020,18:50,5,91,91215,2,1,1,6,VICTOR HUGO (AVENUE),48.697820,2.524460,2020-03-07
4,202000000005,7,3,2020,11:00,1,77,77181,1,6,1,2,LAGNY (RUE DE ) - D35,48.828646,2.705971,2020-03-07


#### Résultat

Le jeu de données `Caract` nettoyé a été correctement exporté dans le dossier `data/processed` sous le nom `caract_2020_2024_clean.csv`.

Il contient les données consolidées et nettoyées des caractéristiques des accidents corporels de la circulation pour la période 2020–2024 et constitue la version prête à être utilisée dans les prochaines étapes du projet.